1. https://huggingface.co/blog/gemma
2. Repo model google/gemma-7b-it is gated. You must be authenticated to access it.
3. Create your 'new token' on your huggingface

pip install torch accelerate transformers langchain chroma sentence-transformers langchain_community pypdf

In [ ]:
#!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
#!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
!which python

In [ ]:
import os
from dotenv import load_dotenv
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GPTQConfig, pipeline
load_dotenv()

In [ ]:
GEMMA_TOKEN = os.getenv("GEMMA_TOKEN")
#https://huggingface.co/google/gemma-3-27b-it
model = "google/gemma-7b-it"
device="cuda"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model, token=GEMMA_TOKEN, device=device)

quantization_config = GPTQConfig(
     bits=4,
     group_size=128,
     dataset="c4", # the original datasets used in GPTQ paper [‘wikitext2’,‘c4’,‘c4-new’,‘ptb’,‘ptb-new’]
     desc_act=False,
     tokenizer=tokenizer,
     batch_size=1,
)
# Define the path to your local model directory
local_model_path = os.path.expanduser("~/federatedMLRagGemma/models/gemma7b")

quantized=True
if quantized:
     model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=local_model_path,  # Path to local model,
                                                 token=GEMMA_TOKEN,
                                                 quantization_config=quantization_config,
                                                 device_map=device
                                                 )
else:
    model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=local_model_path, 
                                                token=GEMMA_TOKEN,
                                                torch_dtype=torch.float16,
                                                device_map=device
                                                )
    '''
    model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path="google/gemma-7b-it",
                                                 token=GEMMA_TOKEN,
                                                 quantization_config=quantization_config,
                                                 device_map=device
                                                 )
    
else:
    model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path="google/gemma-7b-it", 
                                                token=GEMMA_TOKEN,
                                                torch_dtype=torch.float16,
                                                device_map=device
                                                )
'''

In [ ]:
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
messages = [
    {"role": "user", "content": "tell me a joke"},
]
prompt = pipeline.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
outputs = pipeline(
    prompt,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.95
)
print(outputs[0]["generated_text"][len(prompt):])